# Can river-network topology improve wetted-width prediction?

**Target.** Median GRWL wetted width in metres near mean annual flow, matched to a HydroRIVERS reach. The results apply to successfully matched, wider river channels.

**Question.** Does a directed GNN improve width prediction on an unseen basin over a hydraulic power law and tabular models? Each reach is a node. The `NEXT_DOWN` link carries messages from upstream to downstream.

**Run in Colab.** Select **Runtime → Change runtime type → T4 GPU**, then run the cells in order. The default run holds out each of the six basins in turn. Set `QUICK_CHECK = True` for a one-fold check with fewer training epochs, then switch it back to `False` for the results. The last cell downloads `river_width_results.zip`.

Data sources: [HydroRIVERS baseline bundle](https://github.com/AbhinavJain27/BTP-Project/releases/tag/v2-data_bundle), [Allen and Pavelsky's GRWL summary shapefile](https://zenodo.org/records/1297434), and [Frasson et al. on width reliability](https://doi.org/10.1029/2019GL082027).

In [ ]:
%pip -q install "geopandas>=1,<2" "shapely>=2,<3" "pyogrio>=0.9" "pyarrow>=14" "scikit-learn>=1.4,<2" "xgboost>=2,<4"

In [ ]:
from pathlib import Path, PurePosixPath
import gc
import hashlib
import json
import shutil
from zipfile import ZipFile

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyogrio
from pyproj import CRS
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import shapely
from shapely import STRtree
from IPython.display import display

WORK = Path.cwd() / "river_width_experiment"
WORK.mkdir(exist_ok=True)
# Optionally upload your local bundle to Colab and put its path here.
LOCAL_BUNDLE = None
LOCAL_GRWL = None
QUICK_CHECK = False

BASINS = {
    "amazon": "global/amazon", "ganga": "india/ganga", "godavari": "india/godavari",
    "krishna": "india/krishna", "nile": "global/nile", "yangtze": "global/yangtze",
}
BUNDLE_URL = "https://github.com/AbhinavJain27/BTP-Project/releases/download/v2-data_bundle/deploy_bundle.zip"
GRWL_FILE = "GRWL_summaryStats_V01.01.zip"
GRWL_URL = f"https://zenodo.org/records/1297434/files/{GRWL_FILE}?download=1"
GRWL_MD5 = "c37021e144fbff19ae3fc4e701d85d38"

MIN_WIDTH_M = 90.0
MAX_DISTANCE_M = 300.0
MIN_NEAREST_GAP_M = 50.0
SAMPLE_FRACTIONS = (0.2, 0.5, 0.8)
MIN_BASIN_LABELS = 30
FEATURE_FIELDS = ["UPLAND_SKM", "DIS_AV_CMS", "CATCH_SKM", "LENGTH_KM",
                  "DIST_DN_KM", "DIST_UP_KM", "ORD_STRA"]
LOG_FIELDS = FEATURE_FIELDS[:-1]
FEATURE_NAMES = ["log1p_" + f if f in LOG_FIELDS else f for f in FEATURE_FIELDS]
KEY_FIELDS = ["HYRIV_ID", "NEXT_DOWN"]

def checksum(path, algorithm):
    digest = hashlib.new(algorithm)
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def download(url, destination):
    if destination.is_file():
        return destination
    temporary = destination.with_suffix(".part")
    with requests.Session() as session:
        session.mount("https://", HTTPAdapter(max_retries=Retry(
            total=4, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504]
        )))
        with session.get(url, stream=True, timeout=(20, 180)) as response:
            response.raise_for_status()
            with temporary.open("wb") as output:
                for chunk in response.iter_content(1024 * 1024):
                    output.write(chunk)
    if not temporary.is_file() or temporary.stat().st_size == 0:
        raise RuntimeError(f"Empty download: {url}")
    temporary.replace(destination)
    return destination

def extract_reaches(bundle, directory):
    directory.mkdir(exist_ok=True)
    paths = {}
    with ZipFile(bundle) as archive:
        for basin, subdir in BASINS.items():
            member = f"{subdir}/processed/extracted_network.parquet"
            output = directory / f"{basin}.parquet"
            with archive.open(member) as source, output.open("wb") as target:
                shutil.copyfileobj(source, target)
            paths[basin] = output
    return paths

def load_grwl(archive_path, directory):
    directory.mkdir(exist_ok=True)
    with ZipFile(archive_path) as archive:
        shapes = [name for name in archive.namelist() if name.lower().endswith(".shp")
                  and "__MACOSX" not in PurePosixPath(name).parts]
        if len(shapes) != 1:
            raise ValueError(f"Expected one GRWL summary shapefile; found {shapes}")
        shapefile = PurePosixPath(shapes[0])
        for name in archive.namelist():
            member = PurePosixPath(name)
            if member.parent == shapefile.parent and member.stem == shapefile.stem and member.suffix.lower() in {
                ".shp", ".shx", ".dbf", ".prj", ".cpg",
            }:
                with archive.open(name) as source, (directory / member.name).open("wb") as target:
                    shutil.copyfileobj(source, target)
    path = directory / shapefile.name
    names = {name.lower().rstrip("_"): name for name in pyogrio.read_info(path)["fields"]}
    cols = [names["width_med"], names["lakeflag"]]
    grwl = pyogrio.read_dataframe(path, columns=cols, fid_as_index=True)
    grwl = grwl.rename(columns={cols[0]: "width_m", cols[1]: "lakeflag"})
    if grwl.crs is None:
        grwl = grwl.set_crs(4326)  # GRWL summary is documented as WGS 84.
    grwl = grwl.to_crs(4326)
    grwl["grwl_segment"] = grwl.index.astype("int64")
    grwl["width_m"] = pd.to_numeric(grwl["width_m"], errors="coerce")
    grwl["lakeflag"] = pd.to_numeric(grwl["lakeflag"], errors="coerce")
    grwl = grwl.loc[grwl.geometry.notna() & ~grwl.geometry.is_empty].reset_index(drop=True)
    return grwl

## Match satellite labels to river reaches

The match is intentionally conservative. Three interior points on each HydroRIVERS reach vote for a GRWL segment. A point votes only if its nearest segment is within the distance limit and has a clear lead over the second-nearest segment. A label requires two votes for the same segment. Matching considers all GRWL centerlines first, then filters out flagged water bodies and narrow widths. This avoids snapping a narrow reach to a nearby wide river merely because the wide river passes the target filter.

Distances are assessed in a local metre-based projection for each basin. Braided channels, parallel reaches, changes since image collection, and inaccuracies in the simplified GRWL flags can still cause errors. Any GRWL segment assigned to more than one basin is excluded from all folds.

In [ ]:
def match_reaches(reaches, grwl, chunk_size=20_000):
    min_x, min_y, max_x, max_y = reaches.total_bounds
    local = CRS.from_proj4(
        f"+proj=aeqd +lat_0={(min_y + max_y) / 2} +lon_0={(min_x + max_x) / 2} "
        "+datum=WGS84 +units=m +no_defs"
    )
    candidates = grwl.cx[min_x - 0.05:max_x + 0.05, min_y - 0.05:max_y + 0.05].copy()
    cols = ["HYRIV_ID", "width_m", "grwl_segment", "match_m", "votes"]
    if candidates.empty:
        return pd.DataFrame(columns=cols)
    candidate_geometries = candidates.to_crs(local).geometry.to_numpy()
    tree = STRtree(candidate_geometries)
    matching_ids = reaches["HYRIV_ID"].to_numpy(dtype=np.int64)
    matches = []
    point_count = len(SAMPLE_FRACTIONS)

    for offset in range(0, len(reaches), chunk_size):
        chunk = reaches.iloc[offset:offset + chunk_size]
        geometries = chunk.to_crs(local).geometry.to_numpy()
        valid = ~shapely.is_missing(geometries) & ~shapely.is_empty(geometries)
        if not valid.any():
            continue
        geometries = shapely.line_merge(geometries[valid])
        # A disconnected MultiLineString can have ambiguous interior points.
        merged = shapely.get_type_id(geometries) == 1  # LineString
        if not merged.any():
            continue
        lines = geometries[merged]
        ids = matching_ids[offset:offset + len(chunk)][valid][merged]
        points = np.column_stack([
            shapely.line_interpolate_point(lines, fraction, normalized=True)
            for fraction in SAMPLE_FRACTIONS
        ]).reshape(-1)

        # Search beyond the cutoff so a nearly tied runner-up is still visible.
        p, c = tree.query(points, predicate="dwithin", distance=MAX_DISTANCE_M + MIN_NEAREST_GAP_M)
        distance = shapely.distance(points[p], candidate_geometries[c])
        ranked = pd.DataFrame({"point": p, "candidate": c, "distance": distance})
        if ranked.empty:
            continue
        ranked = ranked.sort_values(["point", "distance", "candidate"])
        first = ranked.drop_duplicates("point").set_index("point")
        second = ranked.loc[ranked.groupby("point").cumcount().eq(1)].set_index("point")["distance"]
        clear = (first["distance"] <= MAX_DISTANCE_M) & (
            second.reindex(first.index).fillna(np.inf) - first["distance"] >= MIN_NEAREST_GAP_M
        )
        votes = first.loc[clear].assign(reach=first.index[clear] // point_count)
        consensus = votes.groupby(["reach", "candidate"], as_index=False).agg(
            votes=("distance", "size"), match_m=("distance", "median")
        )
        consensus = consensus.loc[consensus["votes"] >= 2].copy()
        choice = candidates.iloc[consensus["candidate"].to_numpy(dtype=int)]
        consensus["HYRIV_ID"] = ids[consensus["reach"].to_numpy(dtype=int)]
        for attr in ("width_m", "grwl_segment", "lakeflag"):
            consensus[attr] = choice[attr].to_numpy()
        keep = consensus["lakeflag"].eq(0) & np.isfinite(consensus["width_m"]) & consensus["width_m"].ge(MIN_WIDTH_M)
        matches.append(consensus.loc[keep, cols])

    return pd.concat(matches, ignore_index=True) if matches else pd.DataFrame(columns=cols)

def remove_cross_basin_duplicates(labels):
    shared = labels.groupby("grwl_segment")["basin"].nunique().gt(1)
    return labels.loc[~labels["grwl_segment"].isin(shared.index[shared])].copy()

bundle = Path(LOCAL_BUNDLE) if LOCAL_BUNDLE else download(BUNDLE_URL, WORK / "hydrorivers.zip")
grwl_archive = Path(LOCAL_GRWL) if LOCAL_GRWL else download(GRWL_URL, WORK / GRWL_FILE)
if checksum(grwl_archive, "md5") != GRWL_MD5:
    raise ValueError("GRWL ZIP checksum differs from the official summary ZIP. Check the input file.")
reach_paths = extract_reaches(bundle, WORK / "reaches")
grwl = load_grwl(grwl_archive, WORK / "grwl")
parts = []
reach_counts = {}
for basin, path in reach_paths.items():
    reaches = gpd.read_parquet(path, columns=["HYRIV_ID", "geometry"]).to_crs(4326)
    reach_counts[basin] = len(reaches)
    matches = match_reaches(reaches, grwl)
    parts.append(matches.assign(basin=basin))
    print(f"{basin}: {len(matches):,} matched reaches before the cross-basin check", flush=True)
    del reaches
    gc.collect()

raw_labels = pd.concat(parts, ignore_index=True)
labels = remove_cross_basin_duplicates(raw_labels)
if labels.empty:
    raise RuntimeError("No reaches passed the label-quality rules.")
if labels.duplicated(["basin", "HYRIV_ID"]).any():
    raise ValueError("A reach has multiple assigned labels.")
coverage = pd.DataFrame({"basin": list(BASINS), "total_reaches": list(reach_counts.values())}).set_index("basin")
coverage["before_cross_basin_check"] = raw_labels.groupby("basin").size().reindex(coverage.index, fill_value=0)
coverage["matched_reaches"] = labels.groupby("basin").size().reindex(coverage.index, fill_value=0)
coverage["unique_grwl_segments"] = labels.groupby("basin")["grwl_segment"].nunique().reindex(coverage.index, fill_value=0)
coverage["median_match_distance_m"] = labels.groupby("basin")["match_m"].median()
coverage["median_width_m"] = labels.groupby("basin")["width_m"].median()
display(coverage)
del grwl, parts, raw_labels
gc.collect()

## Build the directed graphs

Every model uses the same eligible labels. Observed width, basin/reach IDs, matching distance and GRWL fields are excluded from predictors. `ORD_FLOW` is also excluded. Local inputs are discharge, drainage areas, reach length, distances to the headwater and outlet, and Strahler order. The graph-aware tabular baseline adds direct upstream reach count, sum of discharge and drainage area, and maximum upstream Strahler order.

Two upstream message-passing layers only need the labelled reaches and their two-hop upstream neighbours. The notebook keeps exactly those rows for the GNN. Neighbours contribute input features regardless of whether they have a width label.

In [ ]:
def prepare_basin(path, basin_labels):
    frame = pd.read_parquet(path, columns=KEY_FIELDS + FEATURE_FIELDS)
    reach_ids = frame["HYRIV_ID"].to_numpy(dtype=np.int64)
    index = pd.Index(reach_ids, name="HYRIV_ID")
    if not index.is_unique:
        raise ValueError("Duplicate reach IDs")
    downstream = frame["NEXT_DOWN"].fillna(0).to_numpy(dtype=np.int64)
    destinations = index.get_indexer(downstream)
    internal = destinations >= 0
    if np.any(~internal & (downstream != 0)):
        raise ValueError("The graph has unresolved downstream links.")
    src = np.flatnonzero(internal)
    dst = destinations[internal]
    if np.any(src == dst):
        raise ValueError("The graph has self-loops.")

    raw = frame[FEATURE_FIELDS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    raw[~np.isfinite(raw) | (raw < 0)] = np.nan
    local = raw.copy()
    local[:, :len(LOG_FIELDS)] = np.log1p(local[:, :len(LOG_FIELDS)])
    count = np.bincount(dst, minlength=len(frame))
    total_area = np.bincount(dst, weights=np.nan_to_num(raw[src, 0]), minlength=len(frame))
    total_discharge = np.bincount(dst, weights=np.nan_to_num(raw[src, 1]), minlength=len(frame))
    max_order = np.zeros(len(frame))
    np.maximum.at(max_order, dst, np.nan_to_num(raw[src, -1]))
    upstream = np.column_stack([np.log1p(count), np.log1p(total_area),
                                np.log1p(total_discharge), max_order])

    matched = basin_labels.set_index("HYRIV_ID").reindex(index)
    take = np.flatnonzero(np.isfinite(matched["width_m"].to_numpy(dtype=float)))
    active = np.zeros(len(frame), dtype=bool)
    active[take] = True
    keep_edges = np.zeros(len(src), dtype=bool)
    for _ in range(2):
        keep_edges |= active[dst]
        active[src[keep_edges]] = True
    reindex = np.full(len(frame), -1, dtype=int)
    reindex[active] = np.arange(active.sum())
    result = {
        "local": local[take].astype("float32"),
        "upstream": upstream[take].astype("float32"),
        "observations": matched.iloc[take].reset_index(),
        "x": local[active].astype("float32"),
        "src": reindex[src[keep_edges]], "dst": reindex[dst[keep_edges]],
        "target": reindex[take],
    }
    # Full immediate upstream degree for each active reach. Boundary rows at two hops
    # only contribute their raw features to predictions at the target nodes.
    result["degree"] = count[active].astype("float32")
    if result["observations"]["grwl_segment"].isna().any():
        raise ValueError("Label indexing failed.")
    return result

eligible = [basin for basin in BASINS if coverage.loc[basin, "unique_grwl_segments"] >= MIN_BASIN_LABELS]
if len(eligible) < 3:
    raise RuntimeError("Fewer than three basins have enough independent GRWL segments for a comparison.")
print("Eligible basins:", ", ".join(eligible))
print("Excluded for insufficient label coverage:", ", ".join(set(BASINS) - set(eligible)) or "none")
graphs = {
    basin: prepare_basin(reach_paths[basin], labels.loc[labels["basin"].eq(basin)])
    for basin in eligible
}
for basin, graph in graphs.items():
    print(f"{basin}: {len(graph['x']):,} GNN nodes, {len(graph['src']):,} edges, {len(graph['target']):,} targets")

## Train and evaluate on unseen basins

Each fold keeps one entire basin out of training and feature scaling. Model settings are fixed before testing. The power law uses discharge and upstream area. The tabular and neural models use the same local variables, with separate comparisons for upstream summaries, directed messages, and a no-edge neural control. No observed width from the held-out basin is provided to a model.

During fitting, GRWL segments are weighted equally within each training basin, and each training basin carries the same total weight. Test metrics use equal GRWL segment weights within a basin. Results include reach counts and unique source segment counts. Wider channels are a selected population, so these scores do not estimate skill on narrow headwaters.

In [ ]:
import platform
import sys
from importlib.metadata import version

import torch
from torch import nn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(4, torch.get_num_threads()))
EPOCHS = 5 if QUICK_CHECK else 80
SEEDS = [17] if QUICK_CHECK else [17, 29, 43]
HIDDEN = 48
HELD_OUT = (["godavari"] if "godavari" in eligible else eligible[:1]) if QUICK_CHECK else eligible
print(f"{DEVICE} | {EPOCHS} epochs | {len(SEEDS)} neural seed(s) | {len(HELD_OUT)} fold(s)", flush=True)

def segment_weights(observations):
    counts = observations.groupby("grwl_segment")["grwl_segment"].transform("size")
    weights = 1 / counts.to_numpy(dtype=np.float64)
    return weights / weights.sum()

def weighted_median(values, weights):
    order = np.argsort(values)
    cumulative = np.cumsum(weights[order])
    return float(values[order[np.searchsorted(cumulative, cumulative[-1] / 2)]])

def target(observations):
    return np.log(observations["width_m"].to_numpy(dtype=np.float64))

def fitted_feature_scaler(training_graphs):
    x = np.concatenate([graph["x"] for graph in training_graphs])
    median = np.zeros(x.shape[1], dtype=np.float32)
    for index in range(x.shape[1]):
        observed = x[np.isfinite(x[:, index]), index]
        if observed.size:
            median[index] = np.median(observed)
    filled = np.where(np.isfinite(x), x, median)
    mean = filled.mean(axis=0)
    scale = filled.std(axis=0)
    scale[scale < 1e-6] = 1.0
    return median, mean, scale

def complete_features(x, scaler):
    median, mean, scale = scaler
    return ((np.where(np.isfinite(x), x, median) - mean) / scale).astype(np.float32)

class DirectedWidthNet(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(features * 2, HIDDEN), nn.Linear(HIDDEN * 2, HIDDEN)])
        self.readout = nn.Linear(HIDDEN, 1)

    def forward(self, features, src, dst, degree, use_edges):
        for layer in self.layers:
            incoming = torch.zeros_like(features)
            if use_edges:
                incoming.index_add_(0, dst, features[src])
                incoming = incoming / degree[:, None].clamp_min(1)
            features = torch.relu(layer(torch.cat((features, incoming), dim=1)))
        return self.readout(features).squeeze(1)

def torch_graph(graph, scaler):
    return {
        "x": torch.as_tensor(complete_features(graph["x"], scaler), device=DEVICE),
        "src": torch.as_tensor(graph["src"], dtype=torch.long, device=DEVICE),
        "dst": torch.as_tensor(graph["dst"], dtype=torch.long, device=DEVICE),
        "degree": torch.as_tensor(graph["degree"], device=DEVICE),
        "target_index": torch.as_tensor(graph["target"], dtype=torch.long, device=DEVICE),
    }

def fit_networks(training_graphs, test_graph, scaler, y_offset, use_edges):
    train = [torch_graph(graph, scaler) for graph in training_graphs]
    test = torch_graph(test_graph, scaler)
    train_labels = [
        torch.as_tensor(target(graph["observations"]) - y_offset, dtype=torch.float32, device=DEVICE)
        for graph in training_graphs
    ]
    train_weights = [
        torch.as_tensor(segment_weights(graph["observations"]), dtype=torch.float32, device=DEVICE)
        for graph in training_graphs
    ]

    def predict(network, graph):
        return network(graph["x"], graph["src"], graph["dst"], graph["degree"], use_edges)[graph["target_index"]]

    runs = []
    for seed in SEEDS:
        torch.manual_seed(seed)
        network = DirectedWidthNet(train[0]["x"].shape[1]).to(DEVICE)
        optimiser = torch.optim.AdamW(network.parameters(), lr=0.005, weight_decay=0.001)
        for _ in range(EPOCHS):
            network.train()
            optimiser.zero_grad(set_to_none=True)
            for graph, labels, weights in zip(train, train_labels, train_weights):
                loss = nn.functional.smooth_l1_loss(predict(network, graph), labels, reduction="none")
                ((loss * weights).sum() / len(train)).backward()
            optimiser.step()
        network.eval()
        with torch.no_grad():
            runs.append(predict(network, test).cpu().numpy() + y_offset)
        variant = "directed GNN" if use_edges else "no-edge MLP"
        print(f"  {variant}, seed {seed}: training complete", flush=True)
    return np.stack(runs)

def run_fold(held_out):
    train_names = [name for name in eligible if name != held_out]
    training_graphs = [graphs[name] for name in train_names]
    test_graph = graphs[held_out]
    scaler = fitted_feature_scaler(training_graphs)
    x = np.concatenate([graph["local"] for graph in training_graphs])
    up = np.concatenate([graph["upstream"] for graph in training_graphs])
    y = np.concatenate([target(graph["observations"]) for graph in training_graphs])
    weights = np.concatenate([segment_weights(graph["observations"]) for graph in training_graphs])
    tree_weights = weights * len(weights) / weights.sum()
    predictions = {}

    # A two-variable hydraulic scaling baseline, using log1p(area) and log1p(discharge).
    power = LinearRegression().fit(complete_features(x, scaler)[:, :2], y, sample_weight=weights)
    predictions["power_law"] = power.predict(complete_features(test_graph["local"], scaler)[:, :2])
    for name, train_x, test_x in (
        ("xgb_local", x, test_graph["local"]),
        ("xgb_upstream", np.column_stack((x, up)), np.column_stack((test_graph["local"], test_graph["upstream"]))),
    ):
        model = XGBRegressor(
            n_estimators=100 if QUICK_CHECK else 400, max_depth=5, learning_rate=0.05,
            subsample=0.85, colsample_bytree=0.85, tree_method="hist",
            objective="reg:squarederror", n_jobs=4, random_state=17,
        )
        model.fit(train_x, y, sample_weight=tree_weights)
        predictions[name] = model.predict(test_x)

    y_offset = np.average(y, weights=weights)
    for model_name, use_edges in (("mlp_no_edges", False), ("gnn_directed", True)):
        runs = fit_networks(training_graphs, test_graph, scaler, y_offset, use_edges)
        predictions[model_name] = runs.mean(axis=0)
        for seed, run in zip(SEEDS, runs):
            predictions[f"{model_name}_seed_{seed}"] = run

    output = test_graph["observations"].copy()
    for name, log_widths in predictions.items():
        if not np.isfinite(log_widths).all():
            raise RuntimeError(f"{held_out}: {name} produced non-finite predictions")
        output[name] = np.exp(log_widths)
    return output

MODELS = ["power_law", "xgb_local", "xgb_upstream", "mlp_no_edges", "gnn_directed"]

def score_group(rows, models, by_class=False):
    if rows.empty:
        return []
    weights = segment_weights(rows)
    actual = rows["width_m"].to_numpy(dtype=float)
    scores = []
    for model in models:
        predicted = rows[model].to_numpy(dtype=float)
        absolute = np.abs(predicted - actual)
        log_error = np.log(predicted / actual)
        scores.append({
            "model": model, "reaches": len(rows), "grwl_segments": rows["grwl_segment"].nunique(),
            "mae_m": float(np.average(absolute, weights=weights)),
            "median_absolute_error_m": weighted_median(absolute, weights),
            "median_error_factor": float(np.exp(weighted_median(np.abs(log_error), weights))),
            "median_prediction_to_observation_ratio": float(np.exp(weighted_median(log_error, weights))),
            "log_r2": None if by_class or np.ptp(actual) == 0 else float(
                r2_score(np.log(actual), np.log(predicted), sample_weight=weights)
            ),
        })
    return scores

suffix = "_quick" if QUICK_CHECK else ""
RESULTS = WORK / f"results{suffix}"
RESULTS.mkdir(exist_ok=True)
fold_outputs = []
for basin in HELD_OUT:
    print(f"Training with {basin} held out", flush=True)
    output = run_fold(basin)
    output.insert(0, "held_out_basin", basin)
    fold_outputs.append(output)
    output.to_parquet(RESULTS / f"predictions_{basin}.parquet", index=False)
    display(pd.DataFrame(score_group(output, MODELS)).set_index("model").round(3))
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## Compare error and export results

Use the basin-wise scores before looking at the across-basin average. A GNN result is useful if it beats both tabular baselines across several held-out basins and improves on the no-edge neural model. Also check whether any improvement holds for the narrowest validated channels. Near-one multiplicative error is better. Near-one median prediction/observation ratio indicates low median bias. An error-factor score of 1.5 means a typical prediction is within a factor of 1.5 of the reference.

In [ ]:
predictions = pd.concat(fold_outputs, ignore_index=True)
predictions["width_class_m"] = pd.cut(
    predictions["width_m"], [0, 150, 300, 600, np.inf],
    labels=["<150", "150-299", "300-599", "600+"], right=False, include_lowest=True,
)
basin_scores = []
class_scores = []
seed_scores = []
seed_models = [f"{name}_seed_{seed}" for name in ("mlp_no_edges", "gnn_directed") for seed in SEEDS]
for basin, rows in predictions.groupby("held_out_basin", sort=False):
    basin_scores.extend({"basin": basin, **row} for row in score_group(rows, MODELS))
    seed_scores.extend({"basin": basin, **row} for row in score_group(rows, seed_models))
    for width_class, section in rows.groupby("width_class_m", observed=True):
        class_scores.extend(
            {"basin": basin, "width_class_m": str(width_class), **row}
            for row in score_group(section, MODELS, by_class=True)
        )

metrics = pd.DataFrame(basin_scores)
by_class = pd.DataFrame(class_scores)
seed_metrics = pd.DataFrame(seed_scores)
macro = metrics.groupby("model").agg(
    evaluated_basins=("basin", "nunique"),
    mean_basin_mae_m=("mae_m", "mean"),
    mean_basin_median_error_factor=("median_error_factor", "mean"),
    mean_basin_log_r2=("log_r2", "mean"),
).sort_values("mean_basin_mae_m")

coverage.to_csv(RESULTS / "label_coverage.csv")
metrics.to_csv(RESULTS / "metrics_by_basin.csv", index=False)
by_class.to_csv(RESULTS / "metrics_by_width_class.csv", index=False)
seed_metrics.to_csv(RESULTS / "metrics_by_seed.csv", index=False)
macro.to_csv(RESULTS / "summary.csv")
labels.to_parquet(RESULTS / "matched_width_labels.parquet", index=False)
settings = {
    "quick_check": QUICK_CHECK, "training_basins": eligible, "held_out_basins": HELD_OUT,
    "target": "GRWL median wetted width in metres near mean annual flow",
    "bundle_sha256": checksum(bundle, "sha256"), "grwl_md5": checksum(grwl_archive, "md5"),
    "min_width_m": MIN_WIDTH_M, "max_match_distance_m": MAX_DISTANCE_M,
    "min_nearest_gap_m": MIN_NEAREST_GAP_M, "match_points": SAMPLE_FRACTIONS,
    "minimum_unique_segments_per_basin": MIN_BASIN_LABELS, "feature_fields": FEATURE_FIELDS,
    "epochs": EPOCHS, "seeds": SEEDS, "gnn_hidden_width": HIDDEN, "device": str(DEVICE),
    "platform": platform.platform(), "python": sys.version,
    "versions": {name: version(name) for name in
                 ("torch", "xgboost", "scikit-learn", "geopandas", "shapely", "pyogrio", "pyarrow")},
}
(RESULTS / "run_settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")

display(macro.round(3))
display(metrics.pivot(index="basin", columns="model", values="mae_m").round(1))
fig, ax = plt.subplots(figsize=(12, 5))
metrics.pivot(index="basin", columns="model", values="mae_m")[MODELS].plot.bar(ax=ax)
ax.set(title="Width error on held-out basins", ylabel="Mean absolute error (m)", xlabel="")
ax.legend(title="")
fig.tight_layout()
fig.savefig(RESULTS / "held_out_width_error.png", dpi=160)
plt.show()

archive_path = Path(shutil.make_archive(str(Path.cwd() / f"river_width_results{suffix}"), "zip", RESULTS))
print(f"Results ready: {archive_path.name}")

In [ ]:
from google.colab import files

files.download(str(archive_path))